# Phase 1 — Earth Engine extraction from Google Colab

Run the cells top to bottom. **Where things live:**

| Place | What goes there | Persists? |
|---|---|---|
| **Colab machine** (`/content`) | nothing of ours; Python packages only | No, erased when the session ends |
| **Google Drive** `My Drive/Trabalho/APCerrado/` | the repository clone, including `config.yml` (edited here), the legend table, the decoded results in `data/interim/extract/` | Yes |
| **Google Drive** `My Drive/APCerrado_gee_exports/` | raw CSV exports written by Earth Engine (it only accepts a folder *name*, so it creates the folder at the root of My Drive) | Yes |
| **Earth Engine assets** | `projects/ee-barroso2501/assets/zones_upload` (already uploaded) | Yes |

Nothing needs to be uploaded manually to Colab: the code comes from GitHub (or from the repository zip saved in Drive), and the zones are already an Earth Engine asset.

**After restarting the session, always run cell 1 first:** it mounts Drive and moves back into the repository folder.

## 1. Mount Google Drive and go to the project folder

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = '/content/drive/MyDrive/Trabalho/APCerrado'   # project folder in Drive
REPO = f'{BASE}/Priority-Areas-Cerrado-Pantanal'
os.makedirs(BASE, exist_ok=True)
print('Project folder:', BASE)
# After any restart Colab returns to /content: always move back into the repository
if os.path.exists(REPO):
    %cd $REPO
import sys
sys.path.insert(0, f'{REPO}/scripts/gee')   # absolute path, valid whatever the working folder
print('working folder:', os.getcwd())
print('config.yml found:', os.path.exists(f'{REPO}/scripts/gee/config.yml'))

## 2. Get the code into Drive (only the first time, then just update)
- **Option A (repository already pushed to GitHub):** run the cell below. The first run clones; later runs update with `git pull`.
- **Option B (not pushed yet):** copy `Priority-Areas-Cerrado-Pantanal.zip` to `My Drive/Trabalho/APCerrado/` and set `USE_ZIP = True`.

Caution: `git pull` will not overwrite your local edit of `config.yml`. If both changed, git stops and reports a conflict.

In [ ]:
USE_ZIP = False
if USE_ZIP:
    !cd "$BASE" && unzip -oq Priority-Areas-Cerrado-Pantanal.zip && rm -rf Priority-Areas-Cerrado-Pantanal && mv repo Priority-Areas-Cerrado-Pantanal
elif not os.path.exists(REPO):
    !git clone https://github.com/barroso2501/Priority-Areas-Cerrado-Pantanal.git "$REPO"
else:
    # GitHub is the reference copy: make the Drive copy identical to it. Local edits to
    # versioned files (e.g. config.yml) are discarded; data/interim/ (not versioned) is kept.
    !cd "$REPO" && git fetch -q origin && git reset --hard origin/main
%cd $REPO
!git log --oneline | head -3

## 3. Install packages (every new session)

In [ ]:
!pip install -q earthengine-api pyyaml pyarrow
!python tests/test_gee_logic.py   # offline check: encodings and fire-interval algorithm


## 4. Authenticate and discover the MapBiomas assets (closes P7 and P8)
The first cell opens a Google login. Then `00_discover_assets.py`:
1. lists the MapBiomas folders, so you can find the exact IDs of LULC **Collection 11** and Fire **Collection 5**;
2. while `config.yml` still says `VERIFY`, it stops after the listing (that is expected).

**Then edit `config.yml`** (Colab file panel on the left → `drive/MyDrive/Trabalho/APCerrado/Priority-Areas-Cerrado-Pantanal/scripts/gee/config.yml`, double-click to edit, Ctrl+S to save):
- `lulc_asset`: the Collection 11 integration image;
- `fire_monthly_asset`: the Fire Collection 5 monthly image;
- check the band patterns (`classification_{year}`, `burned_monthly_{year}`) against the band names printed.

**Run the discovery cell again.** It now checks the years 1985–2025, lists the class values found and writes `data/reference/col11_class_values_found.csv`. **Send that output to Claude:** the legend table is updated from it before extraction.

In [ ]:
import ee, yaml
cfg = yaml.safe_load(open(f'{REPO}/scripts/gee/config.yml'))
ee.Authenticate()
ee.Initialize(project=cfg['gee_project'])
print('zones features:', ee.FeatureCollection(cfg['zones_asset']).size().getInfo(), '(expected 3308)')

In [ ]:
%run scripts/gee/00_discover_assets.py

## 5. Start the extractions (only after step 4 is complete)
Each script **starts tasks** on Earth Engine and returns quickly. The processing happens on Google's servers. You can follow it at <https://code.earthengine.google.com/tasks>, and you can close Colab while it runs.

Run the `--test` versions first (year 2025 only). When those tasks finish without error, run the full versions.

In [ ]:
%run scripts/gee/10_extract_lulc.py --test
%run scripts/gee/20_extract_fire_monthly.py --test

In [ ]:
# Full runs: 82 LULC tasks (areas + transitions), 41 fire tasks, 1 interval task
%run scripts/gee/10_extract_lulc.py
%run scripts/gee/20_extract_fire_monthly.py
%run scripts/gee/21_extract_fire_intervals.py

## 6. Decode the exports and run QC (after all tasks have finished)
This reads the CSVs from `My Drive/APCerrado_gee_exports/` and writes Parquet tables plus `qc_report.txt` to `data/interim/extract/` inside the project folder. That folder is not versioned in git but stays in Drive.

In [ ]:
%run scripts/gee/90_collect_exports.py --src "/content/drive/MyDrive/APCerrado_gee_exports" --out "$REPO/data/interim/extract"

## 7. Product 1 extractions (D13) — direct transitions and persistence
Run cell 1 (mount + `%cd`) and cell 3 (packages) first, then update the code from GitHub with the cell of step 2.
This starts **5 tasks**: direct transitions 2012→2025, 2012→2018 and 2018→2025, and persistence of natural vegetation for 2012–2025 and 1985–2025. The 1985–2025 persistence task is the heaviest (41 years); if it fails with a memory error, set `tile_scale: 8` in `config.yml` and run only `--only persist` again.
When all tasks are *Completed* at <https://code.earthengine.google.com/tasks>, run the last cell: it decodes everything again and adds the Product 1 checks to `qc_report.txt`.

In [ ]:
!python tests/test_gee_logic.py   # must end with 'product 1 OK'
%run scripts/gee/30_extract_product1.py

In [ ]:
%run scripts/gee/90_collect_exports.py --src "/content/drive/MyDrive/APCerrado_gee_exports" --out "$REPO/data/interim/extract"
print(open(f'{REPO}/data/interim/extract/qc_report.txt').read())

## 8. Zones × municipalities (D13, municipality filter)
Runs in Colab without Earth Engine (it downloads the IBGE 2025 municipal boundaries, ~1 min). It needs `zones_upload.zip` (the file you uploaded to Earth Engine) inside `Priority-Areas-Cerrado-Pantanal/data/interim/` in Drive. The result, `data/derived/zones_municipalities.csv`, is small and public: upload it to GitHub in `data/derived/`.

In [ ]:
!pip install -q geopandas pyogrio
%run scripts/zones_municipalities.py

## What to commit to GitHub (from your computer or from here)
- **Yes:** `scripts/gee/config.yml` (asset IDs), `data/reference/*.csv` (legend, class values found).
- **No:** `data/interim/` (exports, Parquet); it is ignored by `.gitignore` and stays in Drive.